# How do you evaluate results for a business decision?
**Topics:** Offline Evaluation · A/B Test Results · Causal Model Results · Statistical vs Practical Significance · Trust Checklists

---
## Abbreviation Reference

| Abbreviation | Full Name |
|---|---|
| AUC | Area Under the Curve |
| CATE | Conditional Average Treatment Effect |
| CI | Confidence Interval |
| CUPED | Controlled-experiment Using Pre-Experiment Data |
| DiD | Difference-in-Differences |
| IV | Instrumental Variables |
| MDE | Minimum Detectable Effect |
| ML | Machine Learning |
| NPS | Net Promoter Score |
| PSM | Propensity Score Matching |
| RDD | Regression Discontinuity Design |
| RMSE | Root Mean Squared Error |
| SMD | Standardized Mean Difference |
| SRM | Sample Ratio Mismatch |
| SUTVA | Stable Unit Treatment Value Assumption |

---
## 1. Reading offline evaluation results

### The core gap
- Benchmark performance ≠ real-world validity
- A model that scores well on a held-out test set can still fail to deliver business value
- Three questions to ask: Is it statistically significant? Is it practically significant? Is the test set trustworthy?

### Is the improvement statistically significant?

| Check | What to look for | Red flag |
|---|---|---|
| Test type | Paired test (McNemar's, paired t-test across folds) — not unpaired | Unpaired comparison between two independent runs |
| Confidence Interval (CI) | Report 95% CI around the improvement, not just the point estimate | Point estimate only, no uncertainty quantification |
| Multiple comparisons | Was this the pre-specified primary metric, or the best of many? | "We tried 10 metrics and this one was significant" |
| Minimum Detectable Effect (MDE) | Was the evaluation powered to detect the effect size that matters? | MDE set too small → underpowered; set too large → insensitive |

### Is the improvement practically significant?

| Check | What to look for | Red flag |
|---|---|---|
| Effect size vs. business threshold | Does the improvement exceed the minimum lift that justifies cost? | Statistically significant but below break-even threshold |
| Absolute vs. relative improvement | Relative gains ("10% better") hide small absolute gains on low baselines | "10% improvement" on a 0.1% baseline = 0.01% absolute gain |
| Tail performance | Does the model fail catastrophically on any subgroup? | Great average, terrible worst-case |

### Is the test set trustworthy?

| Check | What to look for | Red flag |
|---|---|---|
| Held-out discipline | Test set never used during training, validation, or hyperparameter tuning | Test set was used to select the final model |
| Data recency | Test set reflects current production distribution | Test set is months or years old; distribution has shifted |
| Label quality | Labels in test set are clean and consistent with production labels | Test labels were generated differently from production labels |
| Leakage check | No features derived from the target or from future information | Features computed using data not available at prediction time |

### Slice analysis as a go/no-go input
- Always break performance down by key segments before making a ship decision
- A global gain that hurts a critical segment is usually not shippable
- Segments to always check: new vs. returning users, mobile vs. desktop, geography, user tenure

### Gotchas
- Never report only the best metric across many experiments — pre-specify the primary metric
- A model that wins on AUC can lose on the metric the business actually uses
- Improvement on a stale test set is not evidence of improvement in production

---
## 2. Reading A/B test results

### Step 0: Sample Ratio Mismatch (SRM) check — do this before looking at any metric
- Expected split (e.g. 50/50) vs. actual split must match within noise
- SRM = randomization bug, bot traffic, or logging failure — the result is invalid
- Use a chi-squared test: p < 0.01 on the split → investigate before proceeding

### Was it pre-registered?

| Scenario | How to treat the result |
|---|---|
| Primary metric defined before launch | Treat as confirmatory — can make a ship decision |
| Primary metric chosen after seeing results | Treat as exploratory — needs replication |
| Multiple metrics tested, best one reported | Apply Bonferroni correction or treat as exploratory |

### Is the primary metric improvement real?

| Check | What to look for | Red flag |
|---|---|---|
| Effect size | Is the improvement above your pre-defined MDE? | Improvement below MDE — test was not powered for this effect |
| Confidence Interval (CI) | Does the 95% CI exclude zero? What is the lower bound? | CI includes zero, or lower bound is below business threshold |
| One-sided vs two-sided | Use two-sided unless direction was pre-specified | Switching to one-sided after seeing data to hit p < 0.05 |
| Peeking | Was the result read only at the pre-specified sample size? | Test stopped early when it "looked significant" |

### Did guardrail metrics hold?
- Guardrail metrics must be defined before launch
- Any guardrail regression = pause before shipping, regardless of primary metric result
- Common guardrails: latency, error rate, 7-day retention, revenue per user

### Novelty effect check
- Users behave differently when something is new — effect may decay over time
- Minimum test duration: one full weekly cycle (7 days), ideally two
- If you have longitudinal data: plot the effect by day — a decaying effect is a warning sign

### Controlled-experiment Using Pre-Experiment Data (CUPED) context
- CUPED reduces variance using pre-experiment user behavior as a covariate
- If CUPED was not applied, confidence intervals are wider than necessary — the result may look non-significant when it isn't
- If CUPED was applied, confirm the pre-experiment covariate is not correlated with treatment assignment

### Segment analysis
- Does the effect hold across key subgroups (new vs. returning, mobile vs. desktop)?
- Strong heterogeneity across segments = consider partial rollout to the segments where effect is clear
- Segment analysis is exploratory — do not use it to "rescue" a failed primary metric

### Gotchas
- A significant result with SRM is not a valid result — fix the SRM first
- Novelty effects make short tests overestimate long-run impact
- Segment wins after a global null are hypothesis-generating, not decision-making inputs

---
## 3. Reading causal model results

### The core problem
- Without randomization, an assumption does the work that randomization would do in an A/B test
- That assumption is always partially untestable — you can stress-test it but never fully verify it
- The trustworthiness question is: how much would the assumption need to be violated to overturn the conclusion?

### Universal trust questions — apply to every causal method

| Question | What to check |
|---|---|
| Is the identifying assumption domain-credible? | Argue it qualitatively — is there a plausible violation? |
| Do falsification / placebo tests pass? | Placebo outcome, placebo time period, or placebo cutoff should return zero effect |
| How sensitive is the estimate to hidden confounding? | Use E-value or Rosenbaum bounds — how strong would an unmeasured confounder need to be to explain away the result? |

### Per-method trust checklist

| Method | Core assumption | Key tests | See |
|---|---|---|---|
| Uplift model | Unconfoundedness — no unmeasured confounders | Quintile lift chart monotonic, Qini above diagonal, placebo outcome near zero, propensity overlap check | cml6_uplift_modeling.ipynb |
| Difference-in-Differences (DiD) | Parallel trends — treated and control would have moved together without treatment | Pre-trend flat across multiple periods, placebo treatment periods show zero effect, event study plot | cml3_observational.ipynb |
| Instrumental Variables (IV) | Relevance + exclusion restriction + independence | First-stage F-statistic > 10, overidentification test passes, exclusion restriction argued qualitatively | cml4_identification.ipynb |
| Regression Discontinuity Design (RDD) | No manipulation at the threshold | McCrary density test smooth, covariate balance at cutoff, stable across bandwidths and polynomial orders | cml4_identification.ipynb |
| Propensity Score Matching (PSM) | Conditional ignorability — all confounders observed | Standardized Mean Difference (SMD) < 0.1 post-matching, common support overlap, Rosenbaum bounds (Γ) high | cml3_observational.ipynb |

### When causal results are strong enough to act on
- Identifying assumption is domain-credible with no obvious violation
- All placebo / falsification tests return near-zero effects
- Sensitivity analysis shows a strong confounder would be needed to overturn the result (high E-value or Γ)
- Effect direction and magnitude are consistent across alternative specifications
- Ideally: a small A/B test or natural experiment corroborates the direction

### Gotchas
- A passing falsification test does not prove the assumption holds — it only fails to reject it
- Causal estimates from observational data are always weaker evidence than a well-run A/B test
- Use causal models to inform decisions when A/B is infeasible — not as a replacement when A/B is possible